# grid-rbd quickstart — iiwa14 (numpy backend)

The **register-then-run** UX: register a robot once (parses the URDF,
generates `grid.cuh`, compiles a per-robot `.so`, content-addressed cache),
then call dynamics algorithms many times — all batched over axis 0.

**Compile-time expectation:** iiwa14 (7-DOF fixed-base) is the cheapest robot
— first `register_robot` is *seconds to tens of seconds* of `nvcc`; a kernel
restart + *Run All* is a **cache hit** (no recompile). Humanoids (g1/h1_2)
take *minutes* — pre-warm their cache out-of-band.

Requires: a CUDA GPU + `nvcc` on PATH + `grid_rbd` installed. This notebook
ends in `assert` cells that cross-check GRiD against `RBDReference`, so a green
run validates numbers, not just 'no exception'.

In [ ]:
import numpy as np
from pathlib import Path
import grid_rbd

# Repo-local URDF (hermetic; no network fetch).
URDF = next(p / 'config/robot_assets' / 'iiwa14.urdf' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config/robot_assets' / 'iiwa14.urdf').exists())
assert URDF.exists(), URDF
np.random.seed(0)

## 1. Register the robot (one-time; cache-hit on re-run)

In [ ]:
h = grid_rbd.register_robot('iiwa14_quickstart', urdf_path=str(URDF),
                            floating_base=False, max_batch_size=64)
print(h)
print('num_joints =', h.num_joints, ' num_vel =', h.num_vel,
      ' num_ees =', h.num_ees, ' num_bodies =', h.num_bodies,
      ' max_batch =', h.max_batch)

Registering again under the same name is **idempotent** — it hits the cache
and returns instantly (no `nvcc`).

In [ ]:
import time
t0 = time.time()
h2 = grid_rbd.register_robot('iiwa14_quickstart', urdf_path=str(URDF),
                             floating_base=False, max_batch_size=64)
print(f'second register_robot took {time.time()-t0:.3f}s (cache hit)')
assert h2.num_joints == h.num_joints

## 2. Run dynamics — single sample and a batch on the *same* handle

Every method takes `(B, num_joints)` float32 and returns `(B, ...)`. `B=1`
and `B=64` are the same call — the batch dim is the workhorse.

In [ ]:
NJ = h.num_joints
q1  = np.random.randn(1, NJ).astype(np.float32)
qd1 = np.random.randn(1, NJ).astype(np.float32)
u1  = np.random.randn(1, NJ).astype(np.float32)

c   = h.inverse_dynamics(q1, qd1)               # inverse dynamics bias  (1, NJ)
qdd = h.forward_dynamics(q1, qd1, u1)  # forward dynamics    (1, NJ)
M   = h.crba(q1)                    # mass matrix            (1, NJ, NJ)
Minv = h.minv(q1)                   # M^-1                   (1, NJ, NJ)
ee  = h.end_effector_pose(q1)       # [xyz, rpy]             (1, 6)
print('inverse_dynamics c  :', np.round(c[0], 3))
print('fd  qdd :', np.round(qdd[0], 3))
print('ee pose :', np.round(ee[0], 3))

In [ ]:
# Same handle, B=64 — throughput is the point.
B = 64
qB  = np.random.randn(B, NJ).astype(np.float32)
qdB = np.random.randn(B, NJ).astype(np.float32)
uB  = np.random.randn(B, NJ).astype(np.float32)
qddB = h.forward_dynamics(qB, qdB, uB)
print('batched forward_dynamics:', qddB.shape)
assert qddB.shape == (B, NJ)

## 3. External forces (`f_ext=`) — optional per-body local-frame wrench

`f_ext` is `(B, 6*num_bodies)`, body-major, each body a length-6
`[angular; linear]` wrench in that link's **local frame** (subtracted from
the per-body force, matching pinocchio / `RBDReference`). Omitting it is the
zero-force path.

In [ ]:
NB = h.num_bodies
f_ext = np.zeros((1, 6 * NB), dtype=np.float32)
f_ext[:, 6*(NB-1):6*(NB-1)+6] = [0, 0, 0, 2.0, 0, 0]  # 2N push on the last link
c_fext = h.inverse_dynamics(q1, qd1, f_ext=f_ext)
print('inverse_dynamics with f_ext changed the bias by:',
      np.round(np.abs(c_fext - c).max(), 4))
assert np.abs(c_fext - c).max() > 1e-3

## 4. Validate vs `RBDReference` (this is the smoke-test assertion)

In [ ]:
from URDFParser import URDFParser
from RBDReference import RBDReference
ref = RBDReference(URDFParser().parse(str(URDF), floating_base=False))
TOL = 5e-3

for i in range(B):
    q, qd, u = qB[i].astype(np.float64), qdB[i].astype(np.float64), uB[i].astype(np.float64)
    c_ref, *_ = ref.inverse_dynamics(q, qd, GRAVITY=-9.81)
    assert np.max(np.abs(h.inverse_dynamics(qB[i:i+1], qdB[i:i+1])[0] - c_ref)) < TOL
    M_ref = ref.crba(q)
    assert np.max(np.abs(h.crba(qB[i:i+1])[0] - M_ref)) < TOL
    qdd_ref = ref.forward_dynamics(q, qd, u)
    assert np.max(np.abs(qddB[i] - qdd_ref)) < TOL
print('OK: inverse_dynamics / crba / forward_dynamics match RBDReference within', TOL)